<a href="https://colab.research.google.com/github/jamermj/Nanowires/blob/main/MR_Notebook1_Data_Processing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 1 — Data processing

Upload both raw-data ZIP files. This notebook creates MR+, MR−, MR_average, smoothed curves, derivatives, and R0 summaries.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import zipfile, os, re, shutil, glob
from google.colab import files

# ============================================================
# SETTINGS
# ============================================================

SMOOTH_WINDOW = 15

# Corrected mapping:
# Dataset 1: CH2 = FL2-F, CH3 = GN2-2
# Dataset 2: CH2 = GN2-3, CH3 = FL2C3
DATASET_CONFIGS = {
    "dataset1": {
        "match": ["GN2-2", "FL2-F", "dataset1"],
        "CH2": "FL2F",
        "CH2_label": "FL2-F",
        "CH3": "GN22",
        "CH3_label": "GN2-2"
    },
    "dataset2": {
        "match": ["GN2-3", "FL2C3", "FL2-C-3", "dataset2"],
        "CH2": "GN23",
        "CH2_label": "GN2-3",
        "CH3": "FL2C3",
        "CH3_label": "FL2C3"
    }
}

# Clean old files
for folder in ["data_files", "MR_outputs"]:
    if os.path.exists(folder):
        shutil.rmtree(folder)

input_dir = "data_files"
outdir = "MR_outputs"
os.makedirs(input_dir, exist_ok=True)
os.makedirs(outdir, exist_ok=True)

# Upload both raw-data ZIP files
uploaded = files.upload()

def identify_dataset(zipname):
    z = zipname.lower()
    for key, cfg in DATASET_CONFIGS.items():
        for term in cfg["match"]:
            if term.lower() in z:
                return key
    raise ValueError(f"Could not identify dataset from zip name: {zipname}")

for zipname in uploaded.keys():
    dataset_key = identify_dataset(zipname)
    extract_dir = os.path.join(input_dir, dataset_key)
    os.makedirs(extract_dir, exist_ok=True)
    with zipfile.ZipFile(zipname, 'r') as z:
        z.extractall(extract_dir)
    print(f"Uploaded {zipname} as {dataset_key}")

# ============================================================
# HELPER FUNCTIONS
# ============================================================

def find_data_start(filepath):
    with open(filepath, 'r', errors='ignore') as f:
        lines = f.readlines()
    for i, line in enumerate(lines):
        if line.strip() == "[Data]":
            return i + 1
    raise ValueError(f"[Data] section not found in {filepath}")


def extract_temperature(filename):
    match = re.search(r'MRat\s*(\d+)K', filename)
    return int(match.group(1)) if match else None


def get_R0_Hminus_Hplus(df, sample_col):
    neg = df[df['Field_Oe'] < 0].copy()
    pos = df[df['Field_Oe'] > 0].copy()

    if len(neg) == 0 or len(pos) == 0:
        raise ValueError(
            f"{sample_col}: need both positive and negative fields. "
            f"Found {len(neg)} negative and {len(pos)} positive."
        )

    idx_neg = neg['Field_Oe'].abs().idxmin()
    idx_pos = pos['Field_Oe'].abs().idxmin()

    H_neg = df.loc[idx_neg, 'Field_Oe']
    R_neg = df.loc[idx_neg, sample_col]
    H_pos = df.loc[idx_pos, 'Field_Oe']
    R_pos = df.loc[idx_pos, sample_col]
    R_avg = 0.5 * (R_neg + R_pos)

    return H_neg, R_neg, H_pos, R_pos, R_avg


def add_MR_and_derivatives(df, sample_col, label):
    H_neg, R_neg, H_pos, R_pos, R_avg = get_R0_Hminus_Hplus(df, sample_col)

    # Definitions
    df[f'MR_{label}_plus'] = (df[sample_col] - R_pos) / R_pos
    df[f'MR_{label}_minus'] = (df[sample_col] - R_neg) / R_neg
    df[f'MR_{label}_average'] = (df[sample_col] - R_avg) / R_avg

    # Smooth MR definitions
    for kind in ["plus", "minus", "average"]:
        df[f'MR_{label}_{kind}_smooth'] = (
            df[f'MR_{label}_{kind}']
            .rolling(window=SMOOTH_WINDOW, center=True, min_periods=1)
            .mean()
        )

    # Derivatives use MR_average_smooth
    df[f'dMRdB_{label}'] = np.gradient(df[f'MR_{label}_average_smooth'], df['Field_T'])
    df[f'd2MRdB2_{label}'] = np.gradient(df[f'dMRdB_{label}'], df['Field_T'])

    return {
        'H_minus_Oe': H_neg,
        'R0_minus': R_neg,
        'H_plus_Oe': H_pos,
        'R0_plus': R_pos,
        'R0_average': R_avg
    }

# ============================================================
# PROCESS FILES
# ============================================================

r0_rows = []
skipped_files = []

for dataset_key, cfg in DATASET_CONFIGS.items():
    dataset_dir = os.path.join(input_dir, dataset_key)
    if not os.path.exists(dataset_dir):
        print(f"Skipping {dataset_key}: no folder found.")
        continue

    for root, dirs, filenames in os.walk(dataset_dir):
        for fname in sorted(filenames):
            if "MRat" not in fname:
                continue

            filepath = os.path.join(root, fname)
            T = extract_temperature(fname)
            base = os.path.splitext(fname)[0]

            print("\n" + "="*90)
            print(f"Processing {dataset_key}: {fname}")
            print("="*90)

            try:
                data_start = find_data_start(filepath)
                df = pd.read_csv(filepath, skiprows=data_start)

                df = df.rename(columns={
                    'Magnetic Field (Oe)': 'Field_Oe',
                    'Bridge 2 Resistivity (Ohm)': cfg["CH2"],
                    'Bridge 3 Resistivity (Ohm)': cfg["CH3"]
                })

                required_cols = ['Field_Oe', cfg["CH2"], cfg["CH3"]]
                missing = [c for c in required_cols if c not in df.columns]
                if missing:
                    raise ValueError(f"Missing required columns after renaming: {missing}")

                df = df[required_cols].dropna()
                for col in required_cols:
                    df[col] = pd.to_numeric(df[col], errors='coerce')
                df = df.dropna().reset_index(drop=True)
                df['Field_T'] = df['Field_Oe'] * 1e-4

                print(f"Field range: {df['Field_Oe'].min():.3f} to {df['Field_Oe'].max():.3f} Oe")
                print(f"N negative points: {(df['Field_Oe'] < 0).sum()}")
                print(f"N positive points: {(df['Field_Oe'] > 0).sum()}")

                r_ch2 = add_MR_and_derivatives(df, cfg["CH2"], cfg["CH2"])
                r_ch3 = add_MR_and_derivatives(df, cfg["CH3"], cfg["CH3"])

                print(f"\n{cfg['CH3_label']}: R0−={r_ch3['R0_minus']:.8e}, R0+={r_ch3['R0_plus']:.8e}, R0avg={r_ch3['R0_average']:.8e}")
                print(f"{cfg['CH2_label']}: R0−={r_ch2['R0_minus']:.8e}, R0+={r_ch2['R0_plus']:.8e}, R0avg={r_ch2['R0_average']:.8e}")

                r0_rows.append({
                    'Dataset': dataset_key,
                    'Temperature_K': T,
                    'Filename': fname,
                    f'{cfg["CH3_label"]}_H_minus_Oe': r_ch3['H_minus_Oe'],
                    f'{cfg["CH3_label"]}_R0_minus': r_ch3['R0_minus'],
                    f'{cfg["CH3_label"]}_H_plus_Oe': r_ch3['H_plus_Oe'],
                    f'{cfg["CH3_label"]}_R0_plus': r_ch3['R0_plus'],
                    f'{cfg["CH3_label"]}_R0_average': r_ch3['R0_average'],
                    f'{cfg["CH2_label"]}_H_minus_Oe': r_ch2['H_minus_Oe'],
                    f'{cfg["CH2_label"]}_R0_minus': r_ch2['R0_minus'],
                    f'{cfg["CH2_label"]}_H_plus_Oe': r_ch2['H_plus_Oe'],
                    f'{cfg["CH2_label"]}_R0_plus': r_ch2['R0_plus'],
                    f'{cfg["CH2_label"]}_R0_average': r_ch2['R0_average']
                })

                out_csv = os.path.join(outdir, f"{dataset_key}_{base}_MR_derivatives.csv")
                df.to_csv(out_csv, index=False)

            except Exception as e:
                print(f"Skipping {dataset_key}: {fname}")
                print(f"Error: {e}")
                skipped_files.append({'Dataset': dataset_key, 'Filename': fname, 'Error': str(e)})

# ============================================================
# EXPORT R0 SUMMARY
# ============================================================

if len(r0_rows) > 0:
    r0_table = pd.DataFrame(r0_rows).sort_values(['Dataset', 'Temperature_K'])
    r0_table.to_csv(os.path.join(outdir, "R0_summary_all_datasets.csv"), index=False)
    display(r0_table)
else:
    print("No valid MR files were processed.")

if len(skipped_files) > 0:
    skipped_table = pd.DataFrame(skipped_files)
    skipped_table.to_csv(os.path.join(outdir, "Skipped_files.csv"), index=False)
    display(skipped_table)

Saving dataset1_GN2-2_FL2-F_old.zip to dataset1_GN2-2_FL2-F_old.zip
Saving dataset2_GN2-3_FL2C3.zip to dataset2_GN2-3_FL2C3.zip
Uploaded dataset1_GN2-2_FL2-F_old.zip as dataset1
Uploaded dataset2_GN2-3_FL2C3.zip as dataset2

Processing dataset1: Ch2FL2-F and Ch-3GN2-2_MRat 100K_+-1Tesla.dat
Field range: nan to nan Oe
N negative points: 0
N positive points: 0
Skipping dataset1: Ch2FL2-F and Ch-3GN2-2_MRat 100K_+-1Tesla.dat
Error: FL2F: need both positive and negative fields. Found 0 negative and 0 positive.

Processing dataset1: Ch2FL2-F and Ch-3GN2-2_MRat 100K_+-2Tesla.dat
Field range: -19999.335 to 19999.524 Oe
N negative points: 162
N positive points: 160

GN2-2: R0−=2.72309087e+00, R0+=2.82697920e+00, R0avg=2.77503503e+00
FL2-F: R0−=2.63630472e-02, R0+=3.15902401e-02, R0avg=2.89766436e-02

Processing dataset1: Ch2FL2-F and Ch-3GN2-2_MRat 200K_+-1Tesla.dat
Field range: nan to nan Oe
N negative points: 0
N positive points: 0
Skipping dataset1: Ch2FL2-F and Ch-3GN2-2_MRat 200K_+-1Tesla

In [ ]:
# Zip processed CSV outputs for transfer to the other notebooks
zip_out = "MR_processed_outputs.zip"
with zipfile.ZipFile(zip_out, 'w') as z:
    for root, dirs, filenames in os.walk(outdir):
        for f in filenames:
            fullpath = os.path.join(root, f)
            z.write(fullpath, arcname=os.path.relpath(fullpath, outdir))
files.download(zip_out)